### COCO-Stuff Distributions

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
os.chdir("..")  # go to project root
print(f"cwd: {os.getcwd()}")  # sanity check

from main import instantiate_from_config, DataModuleFromConfig
from omegaconf import OmegaConf
import numpy as np
from tqdm.notebook import tqdm
import torch
import json

cwd: /home/dude/dev/uni/taming-transformers


In [3]:
DATA_CONFIG = """
target: main.DataModuleFromConfig
params:
    num_workers: 0
    batch_size: 16
    train:
      target: taming.data.coco_wds.CocoWDSImagesAndCaptionsTrain
      params:
        size: 296
        crop_size: 256
        onehot_segmentation: true
        use_stuffthing: true
    validation:
      target: taming.data.coco_wds.CocoWDSImagesAndCaptionsValidation
      params:
        size: 256
        crop_size: 256
        onehot_segmentation: true
        use_stuffthing: true
"""
data_cfg = OmegaConf.create(DATA_CONFIG)

In [4]:
data: DataModuleFromConfig = instantiate_from_config(data_cfg)
data.prepare_data()
data.setup()
train_ldr = data.train_dataloader()
val_ldr = data.val_dataloader()

AttributeError: module 'webdataset' has no attribute 'Dataset'

In [ ]:
with open("data/cocostuffthings/labels.txt", "r") as f:
    labels = [line.strip().split(": ")[1] for line in f.readlines()]
print(labels)

#### Pixel-level class distributions
For all images, display class distribution over all pixels

In [ ]:
def calculate_pixel_level_distribution(loader, num_classes):
    class_counts = torch.zeros(num_classes, dtype=torch.int64, device='cpu')
    for batch in tqdm(loader, desc="Calculating pixel distribution"):
        segmentations = batch["segmentation"]
        counts = torch.bincount(
            segmentations.cpu().flatten(), 
            minlength=num_classes
        )
        class_counts += counts

    total_pixels = class_counts.sum().item()
    return class_counts.numpy(), total_pixels

num_classes = len(labels)
train_class_counts, train_total_pixels = calculate_pixel_level_distribution(train_ldr, num_classes)
val_class_counts, val_total_pixels = calculate_pixel_level_distribution(val_ldr, num_classes)

In [ ]:
with open("analysis/pixel_level_distribution.json", "w") as f:
    json.dump({
        "labels": labels,
        "train": {
            "class_counts": train_class_counts.tolist(),
            "total_pixels": train_total_pixels,
        },
        "validation": {
            "class_counts": val_class_counts.tolist(),
            "total_pixels": val_total_pixels,
        }
    }, f, indent=4)

#### Image-level class distributions
For all images, if an image has at least 1 pixel with the class, increment the counter for that class

In [ ]:
for item in train_ldr.dataset:
    print(item.keys())
    break